# XAI Evaluation — LayerCAM · FPN-LayerCAM · ScoreCAM · FPN-ScoreCAM · LIME

**Model:** CheXpert-finetuned DenseNet169  
**Datasets:** 100 ChestX-ray14 test samples + 100 CheXlocalize test samples  

| Metric | Datasets | Requires GT mask |
|---|---|---|
| Pointing Game | CheXlocalize only | Yes |
| IoU @ 0.5 | CheXlocalize only | Yes |
| mIoU (thresholds 0.1–0.9) | CheXlocalize only | Yes |
| Deletion AUC | Both | No |
| Insertion AUC | Both | No |

---
### GT mask source
`gt_annotations_test.json` — radiologist polygon contours rasterised to binary masks.  
10 evaluatable labels (see `LABEL_MAP`). `gt_segmentations_test.json` also provided  
but polygons are used as the primary source for correctness.

### Key implementation notes
- CAMs are **recomputed** here rather than extracted from saved PNG overlays.
- GT masks are rasterised at original image resolution then resized to 224×224.
- Deletion/Insertion: 50 steps, batch size 20 — feasible on 8 GB VRAM.
- All results saved as CSV + summary plots in `EVAL_OUTPUT_DIR`.

## 0. Imports & Reproducibility

In [ ]:
import gc
import json
import random
import warnings
from copy import deepcopy
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from PIL import Image
from skimage.segmentation import slic
from sklearn.linear_model import Ridge
from sklearn.preprocessing import normalize as sk_normalise
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
import torchvision.models as models

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'  GPU  : {torch.cuda.get_device_name(0)}')
    print(f'  VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## 1. Configuration

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
CHEXPERT_CKPT         = Path('./chexpert_output/best_model.pth')

# Manifests from save_samples notebooks
NIH_MANIFEST          = Path('./test_samples/manifest.csv')
CHEX_MANIFEST         = Path('./chexpert_output/test_samples/manifest.csv')

# Image directories (where the 100 saved PNGs/JPGs live)
NIH_IMAGE_DIR         = NIH_MANIFEST.parent
CHEX_IMAGE_DIR        = CHEX_MANIFEST.parent

# CheXlocalize GT annotation files
GT_ANNOTATIONS_PATH   = Path('./gt_annotations_test.json')
GT_SEGMENTATIONS_PATH = Path('./gt_segmentations_test.json')   # kept for reference

# All evaluation outputs go here
EVAL_OUTPUT_DIR       = Path('./xai_evaluation')
EVAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Model labels (CheXpert output space) ──────────────────────────────────────
LABELS: List[str] = [
    'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity',
    'Lung Lesion', 'Edema', 'Consolidation', 'Pneumonia',
    'Atelectasis', 'Pneumothorax', 'Effusion',
    'Pleural Other', 'Fracture', 'Support Devices', 'No Finding',
]
NUM_CLASSES = len(LABELS)

# ── Model label → CheXlocalize GT label mapping ────────────────────────────────
# Only these 10 labels have radiologist polygon masks in gt_annotations_test.json
LABEL_MAP: Dict[str, str] = {
    'Enlarged Cardiomediastinum': 'Enlarged Cardiomediastinum',
    'Cardiomegaly'              : 'Cardiomegaly',
    'Lung Opacity'              : 'Airspace Opacity',   # CheXpert rename
    'Lung Lesion'               : 'Lung Lesion',
    'Edema'                     : 'Edema',
    'Consolidation'             : 'Consolidation',
    'Atelectasis'               : 'Atelectasis',
    'Pneumothorax'              : 'Pneumothorax',
    'Effusion'                  : 'Pleural Effusion',   # CheXpert rename
    'Support Devices'           : 'Support Devices',
    # Pneumonia, Fracture, Pleural Other, No Finding → no GT mask available
}
EVALUATABLE_LABELS = list(LABEL_MAP.keys())

# ── FPN pyramid ───────────────────────────────────────────────────────────────
FPN_LAYERS: List[Tuple[str, float]] = [
    ('features.denseblock1', 0.20),
    ('features.denseblock2', 0.35),
    ('features.denseblock4', 0.45),
]

# ── LIME settings ─────────────────────────────────────────────────────────────
LIME_N_SEGMENTS   = 80
LIME_COMPACTNESS  = 10
LIME_N_SAMPLES    = 256
LIME_TOP_FEATURES = 10

# ── Deletion/Insertion settings ───────────────────────────────────────────────
DEL_INS_STEPS      = 50      # number of perturbation steps
DEL_INS_BATCH      = 20      # masked images per forward pass
BLUR_KERNEL_SIZE   = 51      # Gaussian blur kernel for Insertion baseline

# ── General ───────────────────────────────────────────────────────────────────
IMG_SIZE   = 224
THRESHOLD  = 0.5
SCORECAM_BATCH = 16

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
LAYERCAM_TARGET = 'features.denseblock4'

XAI_METHODS = ['LayerCAM', 'FPN-LayerCAM', 'ScoreCAM', 'FPN-ScoreCAM', 'LIME']

print('Configuration ready.')
print(f'  Evaluatable labels : {len(EVALUATABLE_LABELS)}')
print(f'  XAI methods        : {XAI_METHODS}')

## 2. Model

In [ ]:
def load_model(ckpt_path: Path, num_classes: int = NUM_CLASSES) -> nn.Module:
    model = models.densenet169(weights=None)
    in_feat = model.classifier.in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.5),
        nn.Linear(in_feat, num_classes),
    )
    ckpt  = torch.load(ckpt_path, map_location='cpu')
    state = ckpt.get('state_dict', ckpt)
    model.load_state_dict(state, strict=True)
    model.eval()
    print(f'Loaded checkpoint — epoch {ckpt.get("epoch","?")}, '
          f'val AUC {ckpt.get("val_auc","?")}')
    return model


model     = load_model(CHEXPERT_CKPT).to(DEVICE)
model_cpu = deepcopy(model).to('cpu').eval()   # used by LIME to save VRAM

## 3. Image & Preprocessing Utilities

In [ ]:
PREPROCESS = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


def load_image(path: Path) -> Tuple[torch.Tensor, np.ndarray]:
    """Returns (tensor (1,3,H,W) on DEVICE, img_rgb uint8 (H,W,3))."""
    pil     = Image.open(path).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
    img_rgb = np.array(pil, dtype=np.uint8)
    tensor  = PREPROCESS(pil).unsqueeze(0).to(DEVICE)
    return tensor, img_rgb


def normalise_cam(cam: np.ndarray) -> np.ndarray:
    mn, mx = cam.min(), cam.max()
    if mx - mn < 1e-8:
        return np.zeros_like(cam, dtype=np.float32)
    return ((cam - mn) / (mx - mn)).astype(np.float32)


@torch.no_grad()
def get_probs(model: nn.Module, tensor: torch.Tensor) -> np.ndarray:
    return model(tensor).sigmoid().squeeze(0).cpu().numpy()

## 4. XAI Methods (reused from XAI notebook)

In [ ]:
class ActivationGradientHook:
    def __init__(self, layer: nn.Module) -> None:
        self.activation: Optional[torch.Tensor] = None
        self.gradient:   Optional[torch.Tensor] = None
        self._fwd = layer.register_forward_hook(
            lambda m, i, o: setattr(self, 'activation', o.detach()))
        self._bwd = layer.register_full_backward_hook(
            lambda m, gi, go: setattr(self, 'gradient', go[0].detach()))

    def remove(self) -> None:
        self._fwd.remove(); self._bwd.remove()


def get_layer(model: nn.Module, name: str) -> nn.Module:
    m = model
    for p in name.split('.'):
        m = getattr(m, p)
    return m


# ── LayerCAM ──────────────────────────────────────────────────────────────────
def layercam(model, tensor, class_idx, layer_name=LAYERCAM_TARGET):
    hook = ActivationGradientHook(get_layer(model, layer_name))
    model.zero_grad()
    model(tensor)[0, class_idx].backward()
    act  = hook.activation.squeeze(0)
    grad = hook.gradient.squeeze(0)
    hook.remove()
    cam  = F.relu((act * F.relu(grad)).sum(0)).cpu().numpy()
    return normalise_cam(cv2.resize(cam, (IMG_SIZE, IMG_SIZE),
                                    interpolation=cv2.INTER_LINEAR))


# ── FPN-LayerCAM ──────────────────────────────────────────────────────────────
def fpn_layercam(model, tensor, class_idx):
    fused = np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.float32)
    for layer_name, weight in FPN_LAYERS:
        hook = ActivationGradientHook(get_layer(model, layer_name))
        model.zero_grad()
        model(tensor)[0, class_idx].backward(retain_graph=True)
        act  = hook.activation.squeeze(0)
        grad = hook.gradient.squeeze(0)
        hook.remove()
        cam  = F.relu((act * F.relu(grad)).sum(0)).cpu().numpy()
        cam  = cv2.resize(cam, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
        fused += weight * normalise_cam(cam)
    model.zero_grad()
    return normalise_cam(fused)


# ── ScoreCAM ──────────────────────────────────────────────────────────────────
def scorecam(model, tensor, class_idx, layer_name=LAYERCAM_TARGET,
             batch_size=SCORECAM_BATCH):
    acts = {}
    h = get_layer(model, layer_name).register_forward_hook(
            lambda m, i, o: acts.update({'v': o.detach()}))
    with torch.no_grad():
        model(tensor)
    h.remove()
    activations = acts['v'].squeeze(0)              # (C, h, w)
    n_ch = activations.shape[0]

    with torch.no_grad():
        baseline = model(torch.zeros_like(tensor)).sigmoid()[0, class_idx].item()

    ch_weights = np.zeros(n_ch, dtype=np.float32)
    for bs in range(0, n_ch, batch_size):
        be = min(bs + batch_size, n_ch)
        masked = []
        for c in range(bs, be):
            ch = activations[c].cpu().numpy()
            ch = cv2.resize(ch, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
            mn, mx = ch.min(), ch.max()
            ch = (ch - mn) / (mx - mn + 1e-8)
            masked.append(tensor * torch.from_numpy(ch).float().to(DEVICE)[None, None])
        with torch.no_grad():
            scores = model(torch.cat(masked)).sigmoid()[:, class_idx].cpu().numpy()
        ch_weights[bs:be] = scores - baseline

    wt = torch.from_numpy(ch_weights).to(activations.device)
    cam = F.relu((F.relu(wt)[:, None, None] * activations).sum(0)).cpu().numpy()
    return normalise_cam(cv2.resize(cam, (IMG_SIZE, IMG_SIZE),
                                    interpolation=cv2.INTER_LINEAR))


# ── FPN-ScoreCAM ──────────────────────────────────────────────────────────────
def fpn_scorecam(model, tensor, class_idx):
    fused = np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.float32)
    for layer_name, weight in FPN_LAYERS:
        fused += weight * scorecam(model, tensor, class_idx, layer_name=layer_name)
    return normalise_cam(fused)


# ── LIME ──────────────────────────────────────────────────────────────────────
def lime_cam(model_cpu_ref, img_rgb, class_idx):
    segments = slic(img_rgb, n_segments=LIME_N_SEGMENTS,
                    compactness=LIME_COMPACTNESS, start_label=0)
    n_segs   = segments.max() + 1
    mean_col = img_rgb.mean(axis=(0, 1))
    rng      = np.random.default_rng(SEED)
    Z        = rng.integers(0, 2, size=(LIME_N_SAMPLES, n_segs)).astype(np.float32)
    Z[0, :]  = 1.0

    perturbed = []
    for z in Z:
        pimg = img_rgb.copy().astype(np.float32)
        for sid in range(n_segs):
            if z[sid] == 0:
                pimg[segments == sid] = mean_col
        perturbed.append(PREPROCESS(Image.fromarray(pimg.astype(np.uint8))))

    batch  = torch.stack(perturbed)
    scores = []
    with torch.no_grad():
        for i in range(0, len(batch), 32):
            scores.append(model_cpu_ref(batch[i:i+32]).sigmoid())
    scores = torch.cat(scores).numpy()             # (N, NUM_CLASSES)

    ones   = np.ones((1, n_segs), dtype=np.float32)
    wts    = np.clip((sk_normalise(Z, 'l2') @ sk_normalise(ones, 'l2').T).squeeze(), 0, None)
    ridge  = Ridge(alpha=1.0).fit(Z, scores[:, class_idx], sample_weight=wts)
    coefs  = np.maximum(ridge.coef_, 0)
    top    = np.argsort(coefs)[-LIME_TOP_FEATURES:]
    mc     = np.zeros_like(coefs); mc[top] = coefs[top]

    hmap = np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.float32)
    for sid in range(n_segs):
        hmap[segments == sid] = mc[sid]
    return normalise_cam(hmap)


def compute_all_cams(
    model:     nn.Module,
    model_cpu_ref: nn.Module,
    tensor:    torch.Tensor,
    img_rgb:   np.ndarray,
    class_idx: int,
) -> Dict[str, np.ndarray]:
    """Returns dict of method_name → cam (H, W) in [0,1] for one class."""
    return {
        'LayerCAM'    : layercam(model, tensor, class_idx),
        'FPN-LayerCAM': fpn_layercam(model, tensor, class_idx),
        'ScoreCAM'    : scorecam(model, tensor, class_idx),
        'FPN-ScoreCAM': fpn_scorecam(model, tensor, class_idx),
        'LIME'        : lime_cam(model_cpu_ref, img_rgb, class_idx),
    }

## 5. Ground-Truth Mask Loader (CheXlocalize)

In [ ]:
# Load annotation file once
with open(GT_ANNOTATIONS_PATH) as f:
    GT_ANNOTATIONS: Dict = json.load(f)

print(f'GT annotations loaded: {len(GT_ANNOTATIONS)} images')


def rasterise_polygons(
    img_key:  str,
    gt_label: str,
    out_size: int = IMG_SIZE,
) -> Optional[np.ndarray]:
    """
    Rasterises all polygon contours for (img_key, gt_label) from
    gt_annotations_test.json into a binary mask of shape (out_size, out_size).

    img_key:  e.g. 'patient64745_study1_view1_frontal' (no .jpg extension)
    gt_label: e.g. 'Cardiomegaly'

    Returns:
      np.ndarray uint8 (out_size, out_size) in {0, 1}
      None if this image/label has no polygon annotation.

    NOTE: Polygon coordinates are in [x, y] format at the original image
    resolution stored in 'img_size'. We rasterise at that resolution then
    resize to out_size to match the CAM resolution.
    """
    if img_key not in GT_ANNOTATIONS:
        return None
    entry = GT_ANNOTATIONS[img_key]
    if gt_label not in entry:
        return None

    H_orig, W_orig = entry['img_size']
    mask = np.zeros((H_orig, W_orig), dtype=np.uint8)

    for polygon in entry[gt_label]:            # one or more polygons per label
        pts = np.array(polygon, dtype=np.float32)
        if len(pts) < 3:
            continue
        pts_int = pts.astype(np.int32).reshape(-1, 1, 2)   # (x, y) pairs
        cv2.fillPoly(mask, [pts_int], 1)

    if mask.sum() == 0:
        return None

    # Resize to CAM resolution using nearest-neighbour to preserve binary values
    mask_resized = cv2.resize(mask, (out_size, out_size),
                              interpolation=cv2.INTER_NEAREST)
    return mask_resized.astype(np.uint8)


def filename_to_gt_key(filename: str) -> str:
    """
    Converts saved sample filename → GT annotation lookup key.
    'patient64541_study1_view1_frontal.jpg' → 'patient64541_study1_view1_frontal'
    """
    return Path(filename).stem


# Quick sanity check
test_key  = 'patient64745_study1_view1_frontal'
test_mask = rasterise_polygons(test_key, 'Cardiomegaly')
print(f'Sanity check — Cardiomegaly mask: shape={test_mask.shape}, '
      f'positive={test_mask.sum()}, '
      f'fraction={test_mask.mean():.4f}')

## 6. Mask-Based Metrics (CheXlocalize)

In [ ]:
def pointing_game(cam: np.ndarray, gt_mask: np.ndarray) -> float:
    """
    Pointing Game (Zhang et al. 2016).
    Returns 1.0 if the pixel of maximum CAM activation falls inside the
    GT mask, 0.0 otherwise.
    Ties are broken by checking if ANY max-value pixel hits the mask.
    """
    max_val  = cam.max()
    max_locs = np.argwhere(cam == max_val)     # (K, 2) — (row, col)
    for r, c in max_locs:
        if gt_mask[r, c] == 1:
            return 1.0
    return 0.0


def iou_at_threshold(cam: np.ndarray, gt_mask: np.ndarray,
                     threshold: float = 0.5) -> float:
    """
    Binarise CAM at `threshold`, compute IoU with binary GT mask.
    IoU = |pred ∩ gt| / |pred ∪ gt|
    Returns NaN if union is zero (neither CAM nor GT has any active pixels).
    """
    pred      = (cam >= threshold).astype(np.uint8)
    intersect = (pred & gt_mask).sum()
    union     = (pred | gt_mask).sum()
    return float(intersect) / float(union) if union > 0 else float('nan')


def mean_iou(
    cam:        np.ndarray,
    gt_mask:    np.ndarray,
    thresholds: np.ndarray = np.linspace(0.1, 0.9, 9),
) -> float:
    """
    mIoU: mean of IoU values computed across multiple thresholds.
    Threshold-agnostic — the primary metric used in Saporta et al. 2022
    (CheXlocalize paper) for XAI evaluation.
    NaN thresholds (empty union) are excluded from the mean.
    """
    ious = [iou_at_threshold(cam, gt_mask, t) for t in thresholds]
    valid = [v for v in ious if not np.isnan(v)]
    return float(np.mean(valid)) if valid else float('nan')

## 7. Mask-Free Metrics — Deletion & Insertion AUC

In [ ]:
def _make_blurred_baseline(tensor: torch.Tensor,
                            kernel_size: int = BLUR_KERNEL_SIZE) -> torch.Tensor:
    """
    Creates a heavily blurred version of the input tensor as the
    Insertion baseline (Petsiuk et al. 2018).
    Uses separable Gaussian approximation via average pooling.
    """
    k   = kernel_size | 1   # ensure odd
    pad = k // 2
    blurred = F.avg_pool2d(
        tensor,
        kernel_size=k,
        stride=1,
        padding=pad,
        count_include_pad=False,
    )
    return blurred.detach()


@torch.no_grad()
def deletion_insertion_auc(
    model:      nn.Module,
    tensor:     torch.Tensor,
    cam:        np.ndarray,
    class_idx:  int,
    n_steps:    int = DEL_INS_STEPS,
    batch_size: int = DEL_INS_BATCH,
) -> Tuple[float, float]:
    """
    Computes Deletion AUC and Insertion AUC for one (image, class, CAM) triple.

    Deletion: progressively replace the most-important pixels (highest CAM)
    with the channel mean. Measures how fast the model's confidence drops.
    Good explanation → fast drop → low AUC.

    Insertion: start from a blurred baseline, progressively reveal the
    most-important pixels. Good explanation → fast rise → high AUC.

    Both AUCs computed via trapezoidal rule over n_steps perturbation levels.
    Returns (deletion_auc, insertion_auc).
    """
    H, W         = cam.shape
    n_pixels     = H * W

    # Pixel ranking: most important first (descending CAM value)
    flat_order   = np.argsort(cam.ravel())[::-1]  # indices, high → low
    step_sizes   = np.linspace(0, n_pixels, n_steps + 1, dtype=int)

    # Baseline tensors
    # Deletion baseline: mean-filled image (per-channel mean)
    mean_vals    = tensor.mean(dim=(0, 2, 3), keepdim=True)   # (1,3,1,1)
    del_baseline = mean_vals.expand_as(tensor).clone()

    # Insertion baseline: blurred image
    ins_baseline = _make_blurred_baseline(tensor)

    del_scores = []
    ins_scores = []
    batch_del, batch_ins = [], []
    step_indices = []

    for step_idx, n_removed in enumerate(step_sizes):
        # Deletion: start from original, blank out top-n_removed pixels
        del_img = tensor.clone()
        if n_removed > 0:
            rows = flat_order[:n_removed] // W
            cols = flat_order[:n_removed] % W
            del_img[0, :, rows, cols] = del_baseline[0, :, rows, cols]

        # Insertion: start from blurred, reveal top-n_removed pixels
        ins_img = ins_baseline.clone()
        if n_removed > 0:
            rows = flat_order[:n_removed] // W
            cols = flat_order[:n_removed] % W
            ins_img[0, :, rows, cols] = tensor[0, :, rows, cols]

        batch_del.append(del_img)
        batch_ins.append(ins_img)
        step_indices.append(step_idx)

        if len(batch_del) == batch_size or step_idx == n_steps:
            d_scores = model(torch.cat(batch_del)).sigmoid()[:, class_idx].cpu().numpy()
            i_scores = model(torch.cat(batch_ins)).sigmoid()[:, class_idx].cpu().numpy()
            del_scores.extend(d_scores.tolist())
            ins_scores.extend(i_scores.tolist())
            batch_del, batch_ins = [], []

    del_scores = np.array(del_scores)
    ins_scores = np.array(ins_scores)
    xs         = np.linspace(0.0, 1.0, len(del_scores))

    del_auc = float(np.trapz(del_scores, xs))
    ins_auc = float(np.trapz(ins_scores, xs))
    return del_auc, ins_auc

## 8. Per-Image Evaluation Core

In [ ]:
def evaluate_image(
    model:          nn.Module,
    model_cpu_ref:  nn.Module,
    img_path:       Path,
    gt_label_cols:  Dict[str, float],    # {label: gt_value} from manifest row
    img_key:        Optional[str],        # GT lookup key for CheXlocalize; None for NIH
) -> List[Dict]:
    """
    Runs all 5 XAI methods on every model-detected class for one image.
    Computes all applicable metrics.

    Returns list of result dicts, one per (method, class) combination.
    Each dict contains: filename, method, label, class_idx, prob,
    pointing_game, iou_05, miou, deletion_auc, insertion_auc.
    NaN for metrics that are not applicable (e.g. mask metrics on NIH images).
    """
    tensor, img_rgb = load_image(img_path)
    probs           = get_probs(model, tensor)

    # Evaluate on all predicted-positive classes
    pred_ids = [i for i, p in enumerate(probs) if p >= THRESHOLD]
    if not pred_ids:
        pred_ids = [int(np.argmax(probs))]

    rows = []
    for class_idx in pred_ids:
        label     = LABELS[class_idx]
        prob      = float(probs[class_idx])

        # GT mask (CheXlocalize only, and only for evaluatable labels)
        gt_mask   = None
        if img_key is not None and label in LABEL_MAP:
            gt_label = LABEL_MAP[label]
            gt_mask  = rasterise_polygons(img_key, gt_label)

        # Compute all CAMs for this class
        cams = compute_all_cams(model, model_cpu_ref, tensor, img_rgb, class_idx)

        for method, cam in cams.items():
            # Mask-based metrics
            pg   = pointing_game(cam, gt_mask)     if gt_mask is not None else float('nan')
            iou  = iou_at_threshold(cam, gt_mask)  if gt_mask is not None else float('nan')
            miou = mean_iou(cam, gt_mask)           if gt_mask is not None else float('nan')

            # Mask-free metrics
            del_auc, ins_auc = deletion_insertion_auc(
                model, tensor, cam, class_idx)

            rows.append({
                'filename'     : img_path.name,
                'img_key'      : img_key or '',
                'method'       : method,
                'label'        : label,
                'class_idx'    : class_idx,
                'prob'         : round(prob, 4),
                'gt_available' : gt_mask is not None,
                'pointing_game': round(pg, 4)   if not np.isnan(pg)   else float('nan'),
                'iou_05'       : round(iou, 4)  if not np.isnan(iou)  else float('nan'),
                'miou'         : round(miou, 4) if not np.isnan(miou) else float('nan'),
                'deletion_auc' : round(del_auc, 4),
                'insertion_auc': round(ins_auc, 4),
            })

    torch.cuda.empty_cache()
    gc.collect()
    return rows

## 9. Batch Evaluation Runner

In [ ]:
def run_evaluation(
    model:          nn.Module,
    model_cpu_ref:  nn.Module,
    manifest_path:  Path,
    image_dir:      Path,
    dataset_name:   str,
    has_gt_masks:   bool,
    results_path:   Path,
) -> pd.DataFrame:
    """
    Iterates over all images in manifest_path, runs evaluate_image,
    appends results incrementally to a CSV (resume-friendly),
    and returns the full results DataFrame.

    has_gt_masks: True for CheXlocalize, False for ChestX-ray14.
    """
    manifest = pd.read_csv(manifest_path)

    # Resume: load already-computed results
    if results_path.exists():
        existing = pd.read_csv(results_path)
        done_files = set(existing['filename'].unique())
        print(f'Resuming — {len(done_files)} images already done.')
    else:
        existing   = pd.DataFrame()
        done_files = set()
        # Write header
        pd.DataFrame(columns=[
            'filename','img_key','method','label','class_idx','prob',
            'gt_available','pointing_game','iou_05','miou',
            'deletion_auc','insertion_auc'
        ]).to_csv(results_path, index=False)

    all_rows = []
    skipped  = []

    for _, row in tqdm(manifest.iterrows(), total=len(manifest),
                       desc=f'Eval [{dataset_name}]'):
        fname    = row['filename']
        if fname in done_files:
            continue

        img_path = image_dir / fname
        if not img_path.exists():
            skipped.append(fname)
            continue

        # GT key for CheXlocalize
        img_key = filename_to_gt_key(fname) if has_gt_masks else None

        # Ground-truth label values from manifest
        gt_cols = {
            lbl: float(row[lbl]) if lbl in row.index else 0.0
            for lbl in LABELS
        }

        try:
            result_rows = evaluate_image(
                model, model_cpu_ref, img_path, gt_cols, img_key)
            all_rows.extend(result_rows)

            # Incremental write — survive crashes
            pd.DataFrame(result_rows).to_csv(
                results_path, mode='a', header=False, index=False)

        except Exception as e:
            print(f'  ERROR on {fname}: {e}')
            skipped.append(fname)

    print(f'\n{dataset_name}: {len(manifest)-len(skipped)-len(done_files)} new images evaluated.')
    if skipped:
        print(f'  Skipped: {skipped}')

    return pd.read_csv(results_path)   # read full file including prior runs

## 10. Run — ChestX-ray14 (Deletion + Insertion only)

In [ ]:
nih_results = run_evaluation(
    model          = model,
    model_cpu_ref  = model_cpu,
    manifest_path  = NIH_MANIFEST,
    image_dir      = NIH_IMAGE_DIR,
    dataset_name   = 'ChestX-ray14',
    has_gt_masks   = False,
    results_path   = EVAL_OUTPUT_DIR / 'results_chestxray14.csv',
)
print(f'NIH results shape: {nih_results.shape}')
nih_results.head(3)

## 11. Run — CheXlocalize (all metrics)

In [ ]:
chex_results = run_evaluation(
    model          = model,
    model_cpu_ref  = model_cpu,
    manifest_path  = CHEX_MANIFEST,
    image_dir      = CHEX_IMAGE_DIR,
    dataset_name   = 'CheXlocalize',
    has_gt_masks   = True,
    results_path   = EVAL_OUTPUT_DIR / 'results_chexlocalize.csv',
)
print(f'CheXlocalize results shape: {chex_results.shape}')
chex_results.head(3)

## 12. Summary Tables

In [ ]:
def summary_by_method(
    df:       pd.DataFrame,
    metrics:  List[str],
    title:    str,
) -> pd.DataFrame:
    """Mean ± std of each metric, rows = methods, cols = metrics."""
    rows = []
    for method in XAI_METHODS:
        sub  = df[df['method'] == method]
        row  = {'Method': method}
        for m in metrics:
            vals = sub[m].dropna()
            if len(vals):
                row[m] = f'{vals.mean():.4f} ± {vals.std():.4f}'
            else:
                row[m] = 'N/A'
        rows.append(row)
    tbl = pd.DataFrame(rows).set_index('Method')
    print(f'\n{title}')
    display(tbl)
    return tbl


# CheXlocalize — all five metrics
chex_mask_rows = chex_results[chex_results['gt_available'] == True]
tbl_chex = summary_by_method(
    chex_results,
    ['pointing_game', 'iou_05', 'miou', 'deletion_auc', 'insertion_auc'],
    'CheXlocalize — All Metrics (mean ± std across images × labels)',
)

# NIH — deletion + insertion only
tbl_nih = summary_by_method(
    nih_results,
    ['deletion_auc', 'insertion_auc'],
    'ChestX-ray14 — Deletion & Insertion AUC (mean ± std)',
)

tbl_chex.to_csv(EVAL_OUTPUT_DIR / 'summary_chexlocalize.csv')
tbl_nih.to_csv(EVAL_OUTPUT_DIR  / 'summary_chestxray14.csv')
print('\nSummary tables saved.')

## 13. Per-Label Breakdown (CheXlocalize)

In [ ]:
def per_label_summary(
    df:      pd.DataFrame,
    metric:  str,
    title:   str,
) -> pd.DataFrame:
    """Pivot: rows = labels, cols = methods."""
    sub  = df[df['gt_available'] == True] if metric in ['pointing_game','iou_05','miou'] else df
    tbl  = sub.groupby(['label', 'method'])[metric].mean().unstack('method')
    # Keep only methods that have data
    tbl  = tbl[[m for m in XAI_METHODS if m in tbl.columns]]
    print(f'\n{title}')
    display(tbl.round(4).style
            .highlight_max(axis=1, color='#c6efce')
            .highlight_min(axis=1, color='#ffeb9c')
            .format('{:.4f}', na_rep='N/A')
           )
    return tbl


lbl_pg   = per_label_summary(chex_results, 'pointing_game',
                              'Pointing Game — per label × method (CheXlocalize)')
lbl_iou  = per_label_summary(chex_results, 'iou_05',
                              'IoU@0.5 — per label × method (CheXlocalize)')
lbl_miou = per_label_summary(chex_results, 'miou',
                              'mIoU — per label × method (CheXlocalize)')
lbl_del  = per_label_summary(chex_results, 'deletion_auc',
                              'Deletion AUC — per label × method (CheXlocalize)')
lbl_ins  = per_label_summary(chex_results, 'insertion_auc',
                              'Insertion AUC — per label × method (CheXlocalize)')

# Save
for name, tbl in [('pointing_game', lbl_pg), ('iou_05', lbl_iou),
                   ('miou', lbl_miou), ('deletion_auc', lbl_del),
                   ('insertion_auc', lbl_ins)]:
    tbl.round(4).to_csv(EVAL_OUTPUT_DIR / f'per_label_{name}_chexlocalize.csv')

print('Per-label tables saved.')

## 14. Visualisations

In [ ]:
# ── Plot 1: Mean metric per method — CheXlocalize ─────────────────────────────
def plot_method_comparison(
    df:       pd.DataFrame,
    metrics:  List[str],
    titles:   List[str],
    dataset:  str,
    filepath: Path,
) -> None:
    n = len(metrics)
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 5))
    if n == 1:
        axes = [axes]

    palette = ['#378ADD', '#1D9E75', '#D85A30', '#7F77DD', '#BA7517']
    colour_map = dict(zip(XAI_METHODS, palette))

    for ax, metric, title in zip(axes, metrics, titles):
        means = []
        errs  = []
        for method in XAI_METHODS:
            vals = df[df['method'] == method][metric].dropna()
            means.append(vals.mean() if len(vals) else 0)
            errs.append(vals.std()   if len(vals) else 0)

        bars = ax.bar(XAI_METHODS, means,
                      color=[colour_map[m] for m in XAI_METHODS],
                      yerr=errs, capsize=4, edgecolor='black', linewidth=0.5)
        ax.set_title(title, fontsize=11)
        ax.set_ylabel('Score')
        ax.tick_params(axis='x', rotation=30)
        ax.grid(axis='y', alpha=0.3)
        for bar, mean in zip(bars, means):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + max(errs) * 0.05,
                    f'{mean:.3f}', ha='center', va='bottom', fontsize=8)

    plt.suptitle(f'XAI Method Comparison — {dataset}', fontsize=13)
    plt.tight_layout()
    plt.savefig(filepath, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved → {filepath}')


# CheXlocalize — mask-based
plot_method_comparison(
    chex_results[chex_results['gt_available'] == True],
    ['pointing_game', 'iou_05', 'miou'],
    ['Pointing Game', 'IoU @ 0.5', 'mIoU (0.1–0.9)'],
    'CheXlocalize',
    EVAL_OUTPUT_DIR / 'plot_mask_metrics_chexlocalize.png',
)

# Both datasets — deletion + insertion
for ds_name, ds_df in [('CheXlocalize', chex_results), ('ChestX-ray14', nih_results)]:
    plot_method_comparison(
        ds_df,
        ['deletion_auc', 'insertion_auc'],
        ['Deletion AUC ↓', 'Insertion AUC ↑'],
        ds_name,
        EVAL_OUTPUT_DIR / f'plot_del_ins_{ds_name.lower().replace("-","")}.png',
    )

In [ ]:
# ── Plot 2: Heatmap — mIoU per label × method ─────────────────────────────────
def plot_heatmap(tbl: pd.DataFrame, title: str, filepath: Path,
                 fmt: str = '.3f', cmap: str = 'YlGn') -> None:
    fig, ax = plt.subplots(figsize=(len(tbl.columns) * 1.5 + 2, len(tbl) * 0.55 + 1.5))
    sns.heatmap(
        tbl.astype(float), annot=True, fmt=fmt, cmap=cmap,
        linewidths=0.4, ax=ax, cbar_kws={'shrink': 0.7},
        annot_kws={'size': 9},
    )
    ax.set_title(title, fontsize=12, pad=10)
    ax.set_xlabel('XAI Method')
    ax.set_ylabel('Label')
    plt.tight_layout()
    plt.savefig(filepath, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved → {filepath}')


plot_heatmap(lbl_pg,
             'Pointing Game — per label × method (CheXlocalize)',
             EVAL_OUTPUT_DIR / 'heatmap_pointing_game.png')

plot_heatmap(lbl_miou,
             'mIoU — per label × method (CheXlocalize)',
             EVAL_OUTPUT_DIR / 'heatmap_miou.png')

plot_heatmap(lbl_del,
             'Deletion AUC — per label × method (CheXlocalize)',
             EVAL_OUTPUT_DIR / 'heatmap_deletion_auc.png',
             cmap='YlOrRd_r')   # lower is better → inverted colormap

plot_heatmap(lbl_ins,
             'Insertion AUC — per label × method (CheXlocalize)',
             EVAL_OUTPUT_DIR / 'heatmap_insertion_auc.png')

In [ ]:
# ── Plot 3: Deletion / Insertion curves for each method ───────────────────────
# Re-runs one representative image per method and plots the step curves

@torch.no_grad()
def del_ins_curves(
    model:      nn.Module,
    tensor:     torch.Tensor,
    cam:        np.ndarray,
    class_idx:  int,
    n_steps:    int = DEL_INS_STEPS,
) -> Tuple[np.ndarray, np.ndarray]:
    """Returns (del_curve, ins_curve) — score at each step."""
    H, W        = cam.shape
    n_pixels    = H * W
    flat_order  = np.argsort(cam.ravel())[::-1]
    step_sizes  = np.linspace(0, n_pixels, n_steps + 1, dtype=int)
    mean_vals   = tensor.mean(dim=(0,2,3), keepdim=True).expand_as(tensor).clone()
    blur_base   = _make_blurred_baseline(tensor)
    del_c, ins_c = [], []
    for n in step_sizes:
        d = tensor.clone()
        i = blur_base.clone()
        if n > 0:
            r = flat_order[:n] // W
            c = flat_order[:n] % W
            d[0,:,r,c] = mean_vals[0,:,r,c]
            i[0,:,r,c] = tensor[0,:,r,c]
        del_c.append(model(d).sigmoid()[0, class_idx].item())
        ins_c.append(model(i).sigmoid()[0, class_idx].item())
    return np.array(del_c), np.array(ins_c)


def plot_del_ins_curves(manifest_path: Path, image_dir: Path,
                         dataset: str, filepath: Path) -> None:
    manifest = pd.read_csv(manifest_path)
    # Pick first image that has at least one predicted class
    for _, row in manifest.iterrows():
        img_path = image_dir / row['filename']
        if img_path.exists():
            tensor, img_rgb = load_image(img_path)
            probs = get_probs(model, tensor)
            pred_ids = [i for i, p in enumerate(probs) if p >= THRESHOLD]
            if not pred_ids:
                pred_ids = [int(np.argmax(probs))]
            class_idx = pred_ids[0]
            break

    xs = np.linspace(0, 1, DEL_INS_STEPS + 1)
    palette = {'LayerCAM':'#378ADD','FPN-LayerCAM':'#1D9E75',
               'ScoreCAM':'#D85A30','FPN-ScoreCAM':'#7F77DD','LIME':'#BA7517'}
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

    for method in XAI_METHODS:
        cam = compute_all_cams(model, model_cpu, tensor, img_rgb, class_idx)[method]
        dc, ic = del_ins_curves(model, tensor, cam, class_idx)
        col = palette[method]
        ax1.plot(xs, dc, label=f'{method} (AUC={np.trapz(dc,xs):.3f})',
                 color=col, lw=2)
        ax2.plot(xs, ic, label=f'{method} (AUC={np.trapz(ic,xs):.3f})',
                 color=col, lw=2)

    ax1.set(title=f'Deletion curve — {LABELS[class_idx]}',
            xlabel='Fraction of pixels removed', ylabel='Model confidence')
    ax1.legend(fontsize=8); ax1.grid(alpha=0.3)
    ax2.set(title=f'Insertion curve — {LABELS[class_idx]}',
            xlabel='Fraction of pixels revealed', ylabel='Model confidence')
    ax2.legend(fontsize=8); ax2.grid(alpha=0.3)

    plt.suptitle(f'Deletion & Insertion curves — {dataset} '
                 f'({row["filename"]})', fontsize=11)
    plt.tight_layout()
    plt.savefig(filepath, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved → {filepath}')


plot_del_ins_curves(CHEX_MANIFEST, CHEX_IMAGE_DIR, 'CheXlocalize',
                    EVAL_OUTPUT_DIR / 'del_ins_curves_chexlocalize.png')
plot_del_ins_curves(NIH_MANIFEST, NIH_IMAGE_DIR, 'ChestX-ray14',
                    EVAL_OUTPUT_DIR / 'del_ins_curves_chestxray14.png')

## 15. Final Summary

In [ ]:
print('=' * 72)
print('  XAI Evaluation — Final Summary')
print('=' * 72)

print('\n── CheXlocalize (100 images, mask-based + mask-free) ──────────────')
for method in XAI_METHODS:
    sub    = chex_results[chex_results['method'] == method]
    mask   = sub[sub['gt_available'] == True]
    pg     = mask['pointing_game'].mean()
    iou    = mask['iou_05'].mean()
    miou_v = mask['miou'].mean()
    d_auc  = sub['deletion_auc'].mean()
    i_auc  = sub['insertion_auc'].mean()
    print(f'  {method:<16s}  PG={pg:.3f}  IoU@0.5={iou:.3f}  '
          f'mIoU={miou_v:.3f}  Del={d_auc:.3f}  Ins={i_auc:.3f}')

print('\n── ChestX-ray14 (100 images, mask-free only) ──────────────────────')
for method in XAI_METHODS:
    sub   = nih_results[nih_results['method'] == method]
    d_auc = sub['deletion_auc'].mean()
    i_auc = sub['insertion_auc'].mean()
    print(f'  {method:<16s}  Del={d_auc:.3f}  Ins={i_auc:.3f}')

print(f'\nAll outputs saved to: {EVAL_OUTPUT_DIR.resolve()}')
print('=' * 72)